In [3]:
from util import llm_call, extract_xml

In [ ]:
def generate(prompt: str, input: str, context: str = "") -> tuple[str, str]:
    full_prompt = f"{prompt}\n{context}\nInput: {input}" if context else f"{prompt}\nInput: {input}"
    response = llm_call(full_prompt, model="claude-3-5-haiku-20241022")
    thoughts = extract_xml(response, "thoughts")
    result = extract_xml(response, "response")
    
    print("\n=== GENERATION START ===")
    print(f"Thoughts:\n{thoughts}\n")
    print(f"Generated:\n{result}")
    print("=== GENERATION END ===\n")
    
    return thoughts, result

def evaluate(prompt: str, content: str, input: str) -> tuple[str, str]:
    # Include the original input in the evaluation prompt
    full_prompt = f"{prompt}\nOriginal task: {input}\nContent to evaluate: {content}"
    response = llm_call(full_prompt)
    evaluation = extract_xml(response, "evaluation")
    feedback = extract_xml(response, "feedback")
    
    print("=== EVALUATION START ===")
    print(f"Status: {evaluation}")
    print(f"Feedback: {feedback}")
    print("=== EVALUATION END ===\n")
    
    return evaluation, feedback

def loop(input: str, evaluator_prompt: str, generator_prompt: str) -> tuple[str, list[dict]]:
    memory = []
    chain_of_thought = []
    
    thoughts, result = generate(generator_prompt, input)
    memory.append(result)
    chain_of_thought.append({"thoughts": thoughts, "result": result})
    
    while True:
        # Pass the original input to the evaluator
        evaluation, feedback = evaluate(evaluator_prompt, result, input)
        if evaluation == "PASS":
            return result, chain_of_thought
            
        context = "\n".join([
            "Previous attempts:",
            *[f"- {m}" for m in memory],
            f"\nFeedback: {feedback}"
        ])
        
        thoughts, result = generate(generator_prompt, input, context)
        memory.append(result)
        chain_of_thought.append({"thoughts": thoughts, "result": result})

In [ ]:
evaluator_prompt = """
Evaluate this Stack implementation for:
- All operations must be O(1)
- getMin() works correctly
- Edge cases (empty stack, duplicate values)
- Memory efficiency
- Clean API

You should be evaluating only and not attemping to solve the task.
Output your evaluation concisely in the following format.

<evaluation>PASS or FAIL</evaluation>
<feedback>
What needs improvement and why.
</feedback>
"""

{answer: PASS/FAIL or feedback: "What needs improvement and why."}

generator_prompt = """
Your goal is to complete the task based on <user input>. If there are feedback 
from your previous generations, you should reflect on them to improve your solution

Output your answer concisely in the following format: 

<thoughts>
- How to track minimum...
- Data structure choice...
- Edge cases...
</thoughts>

<response>
[Your code implementation here]
</response>
"""

input = """
<user input>
Implement a Stack with:
1. push(x)
2. pop()
3. getMin()
All operations should be O(1).
</user input>
"""

In [47]:
loop(input, evaluator_prompt, generator_prompt)



=== GENERATION START ===
Thoughts:
- Need to track minimum element at each step
- Use an auxiliary stack to keep track of minimums
- Ensure O(1) time complexity for all operations
- Handle edge cases like empty stack
- Maintain current minimum at each push/pop operation

Generated:
class MinStack:
    def __init__(self):
        self.stack = []
        self.min_stack = []
    
    def push(self, x: int) -> None:
        self.stack.append(x)
        # If min_stack is empty or x is less/equal to current min, push to min_stack
        if not self.min_stack or x <= self.min_stack[-1]:
            self.min_stack.append(x)
    
    def pop(self) -> None:
        if not self.stack:
            return
        
        # If popped element is current minimum, remove from min_stack too
        if self.stack[-1] == self.min_stack[-1]:
            self.min_stack.pop()
        
        self.stack.pop()
    
    def getMin(self) -> int:
        if not self.min_stack:
            return None
        

('class MinStack:\n    def __init__(self):\n        self.stack = []\n        self.min_stack = []\n    \n    def push(self, x: int) -> None:\n        # Always push to main stack\n        self.stack.append(x)\n        \n        # Push to min stack if empty or x is less than or equal to current min\n        # This handles duplicate minimums correctly\n        if not self.min_stack or x <= self.min_stack[-1]:\n            self.min_stack.append(x)\n    \n    def pop(self) -> None:\n        if not self.stack:\n            return\n        \n        # If top of stack is current minimum, remove from min stack\n        if self.stack[-1] == self.min_stack[-1]:\n            self.min_stack.pop()\n        \n        self.stack.pop()\n    \n    def top(self) -> int:\n        if not self.stack:\n            return None\n        return self.stack[-1]\n    \n    def getMin(self) -> int:\n        if not self.min_stack:\n            return None\n        return self.min_stack[-1]',
 [{'thoughts': '- Need to